In [ ]:
import pandas as pd
from scipy import stats

# --- CONFIGURATION ---
# IMPORTANT: Make sure this file path is correct.
# It must be the same location as your Python script, or you must provide the full path.
FILEPATH = '/content/Experimental Research Study on Financial Decision-Making (Responses) - Form Responses 1.csv'

def load_and_clean_data(filepath):
    """
    Loads the messy CSV, renames columns, and converts percentage ranges to numerical values.
    """
    print(f"Loading data from: {filepath}\n")
    try:
        # header=0 reads the first row as the (messy) header
        df = pd.read_csv(filepath, header=0)
    except FileNotFoundError:
        print(f"ERROR: File not found at '{filepath}'.")
        print("Please make sure the .csv file is in the same directory as this script, or update the FILEPATH variable.")
        return None

    # 1. Define the renaming map for the long, messy column names
    rename_map = {
        'Scenario 1: Allocation Task - Accrual/Savings Framing\n\nImagine you successfully consolidated old savings accounts and received a lump-sum of ₹5,000, which represents funds you have been working to accrue for a long-term goal. How would you allocate this amount (in percentages, total must sum to 100%)?\n\nS1 - Conservative Investment (%): (How much of the ₹5,000 would you allocate to a diversified, low-cost index mutual fund?)\nS1 - Luxury Consumption (%): (How much of the ₹5,000 would you spend on a high-end item, or impulsive personal trip/experience?)\nS1 - Speculative Investment (%): (How much of the ₹5,000 would you invest in a single, highly volatile small-cap stock or speculative cryptocurrency?) [S1 - Conservative Investment (%): (Invest in a diversified, low-cost index mutual fund.)]': 'S1_Conservative',
        'Scenario 1: Allocation Task - Accrual/Savings Framing\n\nImagine you successfully consolidated old savings accounts and received a lump-sum of ₹5,000, which represents funds you have been working to accrue for a long-term goal. How would you allocate this amount (in percentages, total must sum to 100%)?\n\nS1 - Conservative Investment (%): (How much of the ₹5,000 would you allocate to a diversified, low-cost index mutual fund?)\nS1 - Luxury Consumption (%): (How much of the ₹5,000 would you spend on a high-end item, or impulsive personal trip/experience?)\nS1 - Speculative Investment (%): (How much of the ₹5,000 would you invest in a single, highly volatile small-cap stock or speculative cryptocurrency?) [S1 - Luxury Consumption (%): (Spend on a high-end item, or impulsive personal trip/experience.)]': 'S1_Luxury',
        'Scenario 1: Allocation Task - Accrual/Savings Framing\n\nImagine you successfully consolidated old savings accounts and received a lump-sum of ₹5,000, which represents funds you have been working to accrue for a long-term goal. How would you allocate this amount (in percentages, total must sum to 100%)?\n\nS1 - Conservative Investment (%): (How much of the ₹5,000 would you allocate to a diversified, low-cost index mutual fund?)\nS1 - Luxury Consumption (%): (How much of the ₹5,000 would you spend on a high-end item, or impulsive personal trip/experience?)\nS1 - Speculative Investment (%): (How much of the ₹5,000 would you invest in a single, highly volatile small-cap stock or speculative cryptocurrency?) [S1 - Speculative Investment (%): (Invest in a single, highly volatile small-cap stock or speculative cryptocurrency.)]': 'S1_Speculative',
        'Scenario 2: Allocation Task - Income Upgrade/Bonus Framing\n\nImagine you successfully negotiated an exceptional performance bonus and received a lump-sum of ₹5,000, which is equivalent to the extra money from a recent income upgrade. How would you allocate this amount (in percentages, total must sum to 100%)?\n\nS2 - Conservative Investment (%): (How much of the ₹5,000 would you allocate to a diversified, low-cost index mutual fund?)\nS2 - Luxury Consumption (%): (How much of the ₹5,000 would you spend on a high-end item, or impulsive personal trip/experience?)\nS2 - Speculative Investment (%): (How much of the ₹5,000 would you invest in a single, highly volatile small-cap stock or speculative cryptocurrency?) [S2 - Conservative Investment (%): (Invest in a diversified, low-cost index mutual fund.)]': 'S2_Conservative',
        'Scenario 2: Allocation Task - Income Upgrade/Bonus Framing\n\nImagine you successfully negotiated an exceptional performance bonus and received a lump-sum of ₹5,000, which is equivalent to the extra money from a recent income upgrade. How would you allocate this amount (in percentages, total must sum to 100%)?\n\nS2 - Conservative Investment (%): (How much of the ₹5,000 would you allocate to a diversified, low-cost index mutual fund?)\nS2 - Luxury Consumption (%): (How much of the ₹5,000 would you spend on a high-end item, or impulsive personal trip/experience?)\nS2 - Speculative Investment (%): (How much of the ₹5,000 would you invest in a single, highly volatile small-cap stock or speculative cryptocurrency?) [S2 - Luxury Consumption (%): (Spend on a high-end item, or impulsive personal trip/experience.)]': 'S2_Luxury',
        'Scenario 2: Allocation Task - Income Upgrade/Bonus Framing\n\nImagine you successfully negotiated an exceptional performance bonus and received a lump-sum of ₹5,000, which is equivalent to the extra money from a recent income upgrade. How would you allocate this amount (in percentages, total must sum to 100%)?\n\nS2 - Conservative Investment (%): (How much of the ₹5,000 would you allocate to a diversified, low-cost index mutual fund?)\nS2 - Luxury Consumption (%): (How much of the ₹5,000 would you spend on a high-end item, or impulsive personal trip/experience?)\nS2 - Speculative Investment (%): (How much of the ₹5,000 would you invest in a single, highly volatile small-cap stock or speculative cryptocurrency?) [S2 - Speculative Investment (%): (Invest in a single, highly volatile small-cap stock or speculative cryptocurrency.)]': 'S2_Speculative'
    }
    df = df.rename(columns=rename_map)

    # 2. Define the mapping from text ranges to numerical midpoints
    percentage_mapping = {
        '0-25%': 12.5,
        '26-50%': 37.5,
        '51-75%': 62.5,
        '76-100%': 87.5
    }

    # 3. Apply the mapping to all 6 allocation columns
    allocation_columns = [
        'S1_Conservative', 'S1_Luxury', 'S1_Speculative',
        'S2_Conservative', 'S2_Luxury', 'S2_Speculative'
    ]
    for col in allocation_columns:
        df[col] = df[col].map(percentage_mapping)

    # 4. Fill any missing allocation data with 0 (assumes 0%)
    df[allocation_columns] = df[allocation_columns].fillna(0)

    print("Data cleaning complete. Renamed columns and converted ranges to numbers.")
    return df

def run_house_money_tests(df):
    """
    Performs Wilcoxon Signed-Rank Tests to check for the House Money Effect.
    This is a non-parametric test for paired data, perfect for this.
    """

    # --- Test 1: Speculative Investment (Core Hypothesis) ---
    # H0 (Null): The median difference between S1_Speculative and S2_Speculative is zero.
    # HA (Alternative): The median of S2_Speculative is GREATER than S1_Speculative.
    # We use alternative='less' because we test S1 < S2.
    stat_spec, p_spec = stats.wilcoxon(df['S1_Speculative'], df['S2_Speculative'], alternative='less')
    print("\n--- Test 1: Speculative Allocation (S1 vs S2) ---")
    print(f"Wilcoxon Statistic: {stat_spec:.2f}, P-value: {p_spec:.4f}")
    if p_spec < 0.05:
        print("Result: SIGNIFICANT. Candidates allocated significantly MORE to speculative investments in Scenario 2 (Bonus).")
        print(">> This SUPPORTS the House Money Effect.")
    else:
        print("Result: NOT SIGNIFICANT. We cannot conclude a difference in speculative allocation.")

    # --- Test 2: Luxury Consumption ---
    # HA: The median of S2_Luxury is GREATER than S1_Luxury.
    stat_lux, p_lux = stats.wilcoxon(df['S1_Luxury'], df['S2_Luxury'], alternative='less')
    print("\n--- Test 2: Luxury Consumption (S1 vs S2) ---")
    print(f"Wilcoxon Statistic: {stat_lux:.2f}, P-value: {p_lux:.4f}")
    if p_lux < 0.05:
        print("Result: SIGNIFICANT. Candidates allocated significantly MORE to luxury consumption in Scenario 2 (Bonus).")
        print(">> This also SUPPORTS the House Money Effect (mental accounting for 'fun money').")
    else:
        print("Result: NOT SIGNIFICANT. We cannot conclude a difference in luxury spending.")

    # --- Test 3: Conservative Investment ---
    # HA: The median of S2_Conservative is LESS than S1_Conservative.
    # We use alternative='greater' because we test S1 > S2.
    stat_cons, p_cons = stats.wilcoxon(df['S1_Conservative'], df['S2_Conservative'], alternative='greater')
    print("\n--- Test 3: Conservative Allocation (S1 vs S2) ---")
    print(f"Wilcoxon Statistic: {stat_cons:.2f}, P-value: {p_cons:.4f}")
    if p_cons < 0.05:
        print("Result: SIGNIFICANT. Candidates allocated significantly LESS to conservative investments in Scenario 2 (Bonus).")
    else:
        print("Result: NOT SIGNIFICANT. We cannot conclude a difference in conservative allocation.")

def run_correlation_analysis(df):
    """
    Calculates the Spearman correlation matrix for the 6 allocation variables.
    Spearman is used because the data is ordinal (from ranges), not truly continuous.
    """
    allocation_columns = ['S1_Conservative', 'S1_Luxury', 'S1_Speculative', 'S2_Conservative', 'S2_Luxury', 'S2_Speculative']

    # Calculate Spearman's correlation
    corr_matrix = df[allocation_columns].corr(method='spearman')

    print("\n--- Spearman Correlation Matrix ---")
    print("This shows how the allocation choices relate to each other (1 = perfect positive correlation, -1 = perfect negative).")
    print(corr_matrix.to_string())

def run_group_comparison_anova(df):
    """
    Performs a One-Way ANOVA to see if 'Investment Experience Level'
    has a significant effect on the *change* in speculative behavior.
    """

    # 1. Create the 'Speculative_Change' variable
    df['Speculative_Change'] = df['S2_Speculative'] - df['S1_Speculative']

    # 2. Clean data: Drop rows where 'Investment Experience Level' is missing
    df_cleaned = df.dropna(subset=['Investment Experience Level', 'Speculative_Change'])

    # 3. Get the unique groups
    groups = df_cleaned['Investment Experience Level'].unique()

    # 4. Create a list of the data for each group
    group_data = []
    print("\n--- One-Way ANOVA: Impact of Experience on Speculation Change ---")
    print("Comparing the (S2-S1) change in speculative allocation across experience levels:")

    for group_name in groups:
        data = df_cleaned['Speculative_Change'][df_cleaned['Investment Experience Level'] == group_name]
        group_data.append(data)
        print(f"  - Group: {group_name} (n={len(data)}), Mean Change: {data.mean():.2f}%")

    # 5. Perform the One-Way ANOVA
    # This test checks if there is a significant difference between the MEANS of these groups.
    if len(group_data) > 1:
        f_stat, p_value = stats.f_oneway(*group_data)
        print(f"\nANOVA F-Statistic: {f_stat:.2f}, P-value: {p_value:.4f}")
        if p_value < 0.05:
            print("Result: SIGNIFICANT. At least one experience level group behaves significantly differently.")
            print(">> This suggests 'Investment Experience' IS a factor in the House Money Effect.")
            print(">> (Run a post-hoc test like Tukey's to find out *which* groups are different).")
        else:
            print("Result: NOT SIGNIFICANT. We cannot conclude that experience level impacts the change in speculative allocation.")
    else:
        print("ANOVA requires at least two groups to compare.")

def main():
    """
    Main function to run the complete analysis pipeline.
    """
    # Load and clean the data
    df = load_and_clean_data(FILEPATH)

    if df is not None:
        # Run the primary hypothesis tests
        run_house_money_tests(df)

        # Run the correlation analysis
        run_correlation_analysis(df)

        # Run the group-based ANOVA
        run_group_comparison_anova(df)

if __name__ == "__main__":
    main()

Loading data from: /content/Experimental Research Study on Financial Decision-Making (Responses) - Form Responses 1.csv

Data cleaning complete. Renamed columns and converted ranges to numbers.

--- Test 1: Speculative Allocation (S1 vs S2) ---
Wilcoxon Statistic: 5.00, P-value: 0.0145
Result: SIGNIFICANT. Candidates allocated significantly MORE to speculative investments in Scenario 2 (Bonus).
>> This SUPPORTS the House Money Effect.

--- Test 2: Luxury Consumption (S1 vs S2) ---
Wilcoxon Statistic: 0.00, P-value: 0.0009
Result: SIGNIFICANT. Candidates allocated significantly MORE to luxury consumption in Scenario 2 (Bonus).
>> This also SUPPORTS the House Money Effect (mental accounting for 'fun money').

--- Test 3: Conservative Allocation (S1 vs S2) ---
Wilcoxon Statistic: 84.00, P-value: 0.1920
Result: NOT SIGNIFICANT. We cannot conclude a difference in conservative allocation.

--- Spearman Correlation Matrix ---
This shows how the allocation choices relate to each other (1 = per